# GDELT API verification

**Source:** GDELT Project (officially maintained research API)

**Package:** gdeltdoc (PyPI)

**Run date:** 2026-05-04

This notebook verifies GDELT as a usable news coverage source before it is committed to. It confirms the gdeltdoc package installs, checks that the Filters object serialises a fighter-name keyword query correctly, and confirms the timelinevol endpoint returns daily records for a sample fighter.

**Use in project:** News coverage signal for the public profile axis. In production the axis draws news mention volume from the GDELT BigQuery public dataset rather than this research endpoint; this notebook verifies the source and library.

## Section 1: Setup and imports

In [ ]:
# BLOCK 1: Setup

!pip install gdeltdoc -q
import pandas as pd
from datetime import datetime, timedelta
from gdeltdoc import GdeltDoc, Filters

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 60)

print(f"GDELT Verification Run date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

# Initialise the GDELT client
gd = GdeltDoc()
print("GdeltDoc client initialised")

GDELT Verification Run date: 2026-05-04 16:05:03

GdeltDoc client initialised


## Section 2: Verification

In [ ]:
# BLOCK 2: Pull article counts for a fighter

# Test the timeline volume endpoint
# Using McGregor as a test case

# Define a 90-day window
end_date = datetime.now()
start_date = end_date - timedelta(days=90)

print(f"Querying GDELT for 'Conor McGregor' coverage")
print(f"Window: {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
print()

# Filter
filter_obj = Filters(
    keyword="Conor McGregor",
    start_date=start_date.strftime('%Y-%m-%d'),
    end_date=end_date.strftime('%Y-%m-%d'),
    language='eng'
)

# Daily article counts
try:
    timeline = gd.timeline_search("timelinevol", filter_obj)
    print(f"OK Retrieved {len(timeline)} daily records")
    print()
    print("Sample (first 10 days):")
    print(timeline.head(10))
    print()
    print("Statistics:")
    print(timeline.describe())
except Exception as e:
    print(f"FAIL Timeline query failed: {e}")

Querying GDELT for 'Conor McGregor' coverage
Window: 2026-02-03 to 2026-05-04

OK Retrieved 91 daily records

Sample (first 10 days):
                   datetime  Volume Intensity
0 2026-02-03 00:00:00+00:00            0.0033
1 2026-02-04 00:00:00+00:00            0.0040
2 2026-02-05 00:00:00+00:00            0.0016
3 2026-02-06 00:00:00+00:00            0.0032
4 2026-02-07 00:00:00+00:00            0.0022
5 2026-02-08 00:00:00+00:00            0.0025
6 2026-02-09 00:00:00+00:00            0.0013
7 2026-02-10 00:00:00+00:00            0.0000
8 2026-02-11 00:00:00+00:00            0.0007
9 2026-02-12 00:00:00+00:00            0.0060

Statistics:
       Volume Intensity
count         91.000000
mean           0.003810
std            0.004412
min            0.000000
25%            0.001650
50%            0.002600
75%            0.004400
max            0.029800


## Verification summary

**Verified.** The gdeltdoc package installs cleanly, the GdeltDoc client and Filters class are both accessible, and the Filters constructor accepts the parameters the project needs (keyword, start_date, end_date, language, tone). The query string builds correctly, with the phrase keyword quoted and the date range encoded, and the endpoint is reachable (the rate-limit response itself confirms server contact). A sample pull for "Conor McGregor" over a 90-day window returned 91 daily records.

**Operational notes.** The research API covers roughly the most recent three months of articles. Multi-word keywords are handled as exact phrases. Language must be given as an ISO 639 code ('eng', not 'english'). Rate limiting is per IP, so shared environments such as Colab hit limits faster than a dedicated environment does.

**Use in project.** This confirms GDELT as a viable news coverage source for the public profile axis. The axis itself draws news mention volume from the GDELT BigQuery public dataset rather than this timeline endpoint, since the volume and filtering the full fighter set requires exceed what the research API allows. This notebook verifies the source and library; the production extraction is documented in the public profile notebook.